# Food Calorie Estimation Pipeline (All Phases)

This single Colab notebook organizes the project into clearly separated blocks for each phase in the pipeline.

## Phase 0 — Setup

In [ ]:
# Uncomment in Colab if dependencies are not preinstalled
# !pip install torch torchvision ultralytics segment-anything open_clip_torch transformers opencv-python matplotlib rapidfuzz pandas requests

from pathlib import Path

for folder in ["data", "models", "outputs", "src"]:
    Path(folder).mkdir(exist_ok=True)

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
except Exception as exc:
    print('Torch not available yet:', exc)

## Phase 1 — Input Pipeline

In [ ]:
import cv2
import numpy as np

try:
    from google.colab import files
except ImportError:
    files = None

def upload_and_read_image():
    """Upload one image in Colab and return (filename, RGB image array)."""
    if files is None:
        raise RuntimeError('This upload helper requires Google Colab. Use a Colab runtime or load images manually in Jupyter.')
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('No image uploaded.')
    filename = next(iter(uploaded))
    img = cv2.imdecode(np.frombuffer(uploaded[filename], np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Could not decode image: {filename}')
    return filename, cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def preprocess_image(image: np.ndarray, target_size=(512, 512)) -> np.ndarray:
    """Resize an image and normalize pixel values to the [0, 1] range."""
    resized = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0

## Phase 2 — Segmentation

In [ ]:
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class Mask:
    binary_mask: np.ndarray
    bbox: Tuple[int, int, int, int]
    confidence: float

class Segmentor:
    def segment(self, image: np.ndarray) -> List[Mask]:
        """Placeholder segmentor: returns one full-image mask for demo flow."""
        h, w = image.shape[:2]
        return [Mask(binary_mask=np.ones((h, w), dtype=np.uint8), bbox=(0, 0, w, h), confidence=1.0)]

## Phase 3 — Classification

In [ ]:
from typing import Tuple

class Classifier:
    def classify(self, crop: np.ndarray) -> Tuple[str, float]:
        """Classify a food crop and return (label, confidence). Placeholder output."""
        return ('unknown food', 0.0)

def crop_from_mask(image: np.ndarray, mask: Mask) -> np.ndarray:
    """Extract a bounding-box crop and zero out pixels outside the mask region."""
    img_h, img_w = image.shape[:2]
    x, y, w, h = mask.bbox
    x = max(0, min(x, img_w))
    y = max(0, min(y, img_h))
    x2 = max(x, min(x + w, img_w))
    y2 = max(y, min(y + h, img_h))
    cropped = image[y:y2, x:x2].copy()
    region = mask.binary_mask[y:y2, x:x2]
    if region.ndim == 2:
        cropped[region == 0] = 0
    return cropped

## Phase 4 — Volume Estimation

In [ ]:
from abc import ABC, abstractmethod

class VolumeEstimator(ABC):
    @abstractmethod
    def estimate(self, mask: Mask, image: np.ndarray) -> float:
        """Estimate food volume in cm³ for a given mask and source image."""

class PlatePriorVolumeEstimator(VolumeEstimator):
    def __init__(self, plate_diameter_cm: float = 26.0):
        self.plate_diameter_cm = plate_diameter_cm

    def estimate(self, mask: Mask, image: np.ndarray) -> float:
        # Placeholder conversion: each positive mask pixel contributes 0.01 cm³.
        area_pixels = float(mask.binary_mask.sum())
        return area_pixels * 0.01

## Phase 5 — Calorie Database

In [ ]:
import pandas as pd

class CSVCalorieDB:
    def __init__(self, csv_path: str = 'data/calorie_db.csv'):
        default_columns = ['label', 'density_g_per_cm3', 'kcal_per_100g']
        if Path(csv_path).exists():
            try:
                self.df = pd.read_csv(csv_path)
            except Exception:
                self.df = pd.DataFrame(columns=default_columns)
        else:
            self.df = pd.DataFrame(columns=default_columns)

        for col in default_columns:
            if col not in self.df.columns:
                if col == 'label':
                    self.df[col] = pd.Series(dtype='object')
                else:
                    self.df[col] = pd.Series(dtype='float64')

    def lookup(self, label: str, volume_cm3: float) -> float:
        """Estimate kcal for a food label and volume; returns 0.0 when not found."""
        if self.df.empty:
            return 0.0
        labels = self.df['label'].fillna('').astype(str).str.lower()
        rows = self.df[labels == label.lower()]
        if rows.empty:
            return 0.0
        row = rows.iloc[0]
        try:
            grams = volume_cm3 * float(row['density_g_per_cm3'])
            return grams * float(row['kcal_per_100g']) / 100.0
        except (TypeError, ValueError):
            return 0.0

## Phase 6 — Evaluation & Debug UI

In [ ]:
import matplotlib.pyplot as plt

def run_pipeline(image: np.ndarray, segmentor: Segmentor, classifier: Classifier, volume_estimator: VolumeEstimator, calorie_db: CSVCalorieDB):
    """Run all pipeline stages and return per-item results with total calories."""
    masks = segmentor.segment(image)
    rows = []
    for m in masks:
        crop = crop_from_mask(image, m)
        label, confidence = classifier.classify(crop)
        volume_cm3 = volume_estimator.estimate(m, image)
        kcal = calorie_db.lookup(label, volume_cm3)
        rows.append({
            'label': label,
            'confidence': confidence,
            'volume_cm3': volume_cm3,
            'kcal': kcal,
        })

    result_df = pd.DataFrame(rows)
    total_kcal = float(result_df['kcal'].sum()) if not result_df.empty else 0.0

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'Estimated Total Calories: {total_kcal:.1f} kcal')
    plt.show()

    return result_df, total_kcal

# Example usage in Colab:
# filename, image = upload_and_read_image()
# image_pp = preprocess_image(image)
# results, total = run_pipeline(image_pp, Segmentor(), Classifier(), PlatePriorVolumeEstimator(), CSVCalorieDB())
# results